# 13 - LangGraph Governed SQL Agent Runtime

This notebook uses a real LangGraph `StateGraph` to model a governed SQL agent. The graph plans a SQL draft, validates it with Metatate, and then routes to one of three outcomes:

- approve safe SQL
- revise SQL that needs minimization
- block SQL for prohibited use

The planner is deterministic so the example is repeatable in offline mode and live mode. In production, an LLM can replace the deterministic planner node while keeping the Metatate validation and routing nodes in the same position.


In [ ]:
from pathlib import Path
import json
import os
import sys

import pandas as pd

repo_root = Path.cwd()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root))

from common import get_client

client = get_client()
mode = os.getenv("METATATE_EXAMPLES_MODE", "offline")
print(f"Metatate examples mode: {mode}")


def decision_label(response):
    data = response.get("data", {})
    decision = data.get("decision")
    if isinstance(decision, dict):
        return decision.get("decision") or decision.get("overall_decision") or data.get("overall_decision", "UNKNOWN")
    return decision or data.get("overall_decision", "UNKNOWN")


def rationale_text(response):
    data = response.get("data", {})
    decision = data.get("decision")
    if isinstance(decision, dict):
        return decision.get("rationale") or data.get("summary") or ""
    return data.get("rationale") or data.get("summary") or ""


def agent_action_text(response):
    action = response.get("data", {}).get("agent_action", {})
    if isinstance(action, dict):
        return action.get("message") or action.get("safe_next_step") or action.get("suggested_next_tool") or ""
    return ""


## Build the graph


In [ ]:
from framework_runtime.langgraph_governed_sql_agent import (
    build_governed_sql_agent,
    summarize_state,
)

app = build_governed_sql_agent(client)
print("LangGraph governed SQL agent compiled.")


## Run representative requests


In [ ]:
questions = {
    "safe": "Show ARR by region for active customers.",
    "unsafe": "Show EU customers and include their emails so analysts can identify accounts.",
    "blocked": "Build a direct marketing email campaign list for active customers.",
}

states = {
    name: app.invoke({"question": question})
    for name, question in questions.items()
}
summaries = {
    name: summarize_state(state)
    for name, state in states.items()
}

pd.DataFrame(summaries).T[
    ["route", "decision", "intended_use", "validation_id", "answer"]
]


## Inspect the SQL routing outcomes


In [ ]:
for name, state in states.items():
    summary = summarize_state(state)
    print(f"=== {name.upper()} ===")
    print(f"Route: {summary['route']}")
    print(f"Decision: {summary['decision']}")
    print(f"Validation ID: {summary['validation_id']}")
    print("Draft SQL:")
    print(summary["draft_sql"])
    print("Final SQL:")
    print(summary["final_sql"])
    print("Notes:")
    print(summary["notes"])
    print()


The graph only returns executable SQL from the approve or revise branches. The block branch returns no SQL, which keeps prohibited requests from leaking into downstream execution.
